In [1]:

!pip install -q timm einops albumentations opencv-python-headless

import os, time, random, copy, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
import cv2
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import albumentations as A
from albumentations.pytorch import ToTensorV2
from einops import rearrange
from sklearn.metrics import *
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  DATASET CONFIGURATIONS WITH CORRECT PATHS AND CLASS ORDERS                 ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

BASE_PATH = '/kaggle/input/datasets/bhagawati4204/daataasets'

DATASET_CONFIGS = {
    'brain_tumor': {
        'name': 'Brain Tumor MRI',
        'class_names': ['notumor', 'glioma', 'meningioma', 'pituitary'],
        'task': 'multi_class',
        'is_grayscale': True,
        'path': f'{BASE_PATH}/Brain Tumor MRI Dataset/Brain Tumor MRI Dataset',
        'use_train_test_split': True  # ✅ Use 70/30 train/test split
    },
    'fracatlas': {
        'name': 'FracAtlas',
        'class_names': ['Not Fractured', 'Fractured'],
        'task': 'binary',
        'is_grayscale': True,
        'path': f'{BASE_PATH}/Fracture Dataset/Fracture Dataset/FracAtlas/images',
        'use_train_test_split': False
    },
    'tb_chest': {
        'name': 'TB Chest X-Ray',
        'class_names': ['Normal', 'Tuberculosis'],
        'task': 'binary',
        'is_grayscale': True,
        'path': f'{BASE_PATH}/Tuberculosis (TB) Chest X-ray Database/Tuberculosis (TB) Chest X-ray Database/TB_Chest_Radiography_Database',
        'use_train_test_split': False
    },
    'pneumonia': {
        'name': 'Pneumonia Chest X-Ray',
        'class_names': ['Normal', 'Pneumonia'],
        'task': 'binary',
        'is_grayscale': True,
        'path': f'{BASE_PATH}/Chest X-Ray Images (Pneumonia)/Chest X-Ray Images (Pneumonia)/chest_xray',
        'use_train_test_split': False
    },
    'diabetic_retinopathy': {
        'name': 'Diabetic Retinopathy',
        'class_names': ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferate_DR'],
        'task': 'multi_class',
        'is_grayscale': False,
        'path': f'{BASE_PATH}/Diabetic Retinopathy 224x224 (2019 Data)/Diabetic Retinopathy 224x224 (2019 Data)/colored_images',
        'use_train_test_split': False
    },
    'bone_fracture': {
        'name': 'Bone Fracture Binary',
        'class_names': ['Not Fractured', 'Fractured'],
        'task': 'binary',
        'is_grayscale': True,
        'path': f'{BASE_PATH}/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification',
        'use_train_test_split': False
    },

    'ham10000': {
        'name': 'HAM10000 Skin Cancer',
        'class_names': ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'],
        'task': 'multi_class',
        'is_grayscale': False,
        'path': f'{BASE_PATH}/skin cancer/skin cancer',
        'use_train_test_split': False
    }
}

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  MODEL ARCHITECTURE (EXACT MATCH TO TRAINING)                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0): 
        super().__init__(); self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training: return x
        return x * (torch.rand((x.shape[0], 1, 1), dtype=torch.float32, device=x.device) * float(1.0 - self.drop_prob))

class WindowAttention(nn.Module):
    def __init__(self, d_model, nhead, window_size=7, dropout=0.1):
        super().__init__()
        self.nhead, self.window_size, self.head_dim = nhead, window_size, d_model // nhead
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop, self.proj_drop, self.attn_weights = nn.Dropout(dropout), nn.Dropout(dropout), None
        
    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.reshape(B, H, W, C)
        pad_h, pad_w = (self.window_size - H % self.window_size) % self.window_size, (self.window_size - W % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        Hp, Wp = x.shape[1], x.shape[2]
        x = x.reshape(B, Hp//self.window_size, self.window_size, Wp//self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(-1, self.window_size**2, C)
        qkv = self.qkv(x).reshape(-1, self.window_size**2, 3, self.nhead, self.head_dim).permute(2,0,3,1,4)
        attn = (qkv[0] @ qkv[1].transpose(-2,-1)) * self.scale
        attn = attn.softmax(dim=-1)
        self.attn_weights = attn.detach()
        x = self.proj_drop(self.proj((self.attn_drop(attn) @ qkv[2]).transpose(1,2).reshape(-1, self.window_size**2, C)))
        x = x.reshape(B, Hp//self.window_size, Wp//self.window_size, self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(B, Hp, Wp, C)
        return x[:, :H, :W, :].reshape(B, -1, C) if pad_h > 0 or pad_w > 0 else x.reshape(B, -1, C)

class TransformerBlockWithWindow(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim, window_size=7, attn_drop=0.1, ffn_drop=0.1, drop_path=0.0):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.global_attn = nn.MultiheadAttention(d_model, nhead, dropout=attn_drop, batch_first=True)
        self.window_attn = WindowAttention(d_model, nhead, window_size, attn_drop)
        self.gate = nn.Sequential(nn.Linear(d_model*2, d_model), nn.Sigmoid())
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(ffn_drop),
            nn.Linear(ffn_dim, d_model), nn.Dropout(ffn_drop)
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.attn_weights, self.window_attn_weights = None, None
        
    def forward(self, x, H, W):
        x_norm = self.norm1(x)
        global_out, global_attn = self.global_attn(x_norm, x_norm, x_norm, need_weights=True)
        self.attn_weights = global_attn.detach()
        window_out = self.window_attn(x_norm[:, 1:, :], H, W)
        self.window_attn_weights = self.window_attn.attn_weights
        gate = self.gate(torch.cat([global_out[:, 1:, :], window_out], dim=-1))
        combined = torch.cat([global_out[:, :1, :], gate * global_out[:, 1:, :] + (1-gate) * window_out], dim=1)
        x = x + self.drop_path(combined)
        return x + self.drop_path(self.ffn(self.norm2(x)))

class ImprovedBackbone(nn.Module):
    def __init__(self): 
        super().__init__(); 
        self.backbone = models.convnext_base(weights=None)
        self.feature_extractor = self.backbone.features
    def forward(self, x): return self.feature_extractor(x)

class EnhancedHViT(nn.Module):
    def __init__(self, num_classes=2, img_size=224, d_model=384, nhead=6, n_layers=4, dim_ffn=1536):
        super().__init__()
        self.d_model = d_model
        self.img_size = img_size
        self.backbone = ImprovedBackbone()
        with torch.no_grad():
            _, c, h, w = self.backbone(torch.zeros(1, 3, img_size, img_size)).shape
        self.spatial_h, self.spatial_w = h, w
        self.proj = nn.Sequential(
            nn.Conv2d(c, d_model, 1, bias=False), nn.BatchNorm2d(d_model), nn.GELU(),
            nn.Conv2d(d_model, d_model, 3, padding=1, bias=False), nn.BatchNorm2d(d_model), nn.GELU()
        )
        self.norm_proj = nn.LayerNorm(d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, h*w + 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        dpr = [x.item() for x in torch.linspace(0, 0.15, n_layers)]
        self.encoder = nn.ModuleList([
            TransformerBlockWithWindow(d_model, nhead, dim_ffn, min(7, h, w), drop_path=dpr[i])
            for i in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model//2), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(d_model//2, d_model//4), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(d_model//4, num_classes)
        )
        
    def forward(self, x):
        B = x.size(0)
        tokens = rearrange(self.proj(self.backbone(x)), 'b d h w -> b (h w) d')
        tokens = self.norm_proj(tokens)
        tokens = torch.cat([self.cls_token.expand(B,-1,-1), tokens], dim=1) + self.pos_embed
        for block in self.encoder:
            tokens = block(tokens, self.spatial_h, self.spatial_w)
        return self.head(self.norm(tokens)[:, 0])
    
    def get_attentions(self):
        g, w = [], []
        for b in self.encoder:
            if b.attn_weights is not None: g.append(b.attn_weights)
            if b.window_attn_weights is not None: w.append(b.window_attn_weights)
        return g, w

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  DATASET CLASS                                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

class MedicalDataset(Dataset):
    def __init__(self, samples, class_names, transform=None, is_grayscale=False):
        self.samples = samples
        self.class_names = class_names
        self.class_to_idx = {n: i for i, n in enumerate(class_names)}
        self.transform = transform
        self.is_grayscale = is_grayscale
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        p, l = self.samples[idx]
        l = self.class_to_idx[l]
        
        try:
            if self.is_grayscale:
                img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    img = np.array(Image.open(p).convert('L'))
                img = np.stack([img, img, img], axis=-1)
            else:
                img = cv2.imread(p)
                if img is None:
                    img = np.array(Image.open(p).convert('RGB'))
                else:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        except:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        
        if self.transform:
            img = self.transform(image=img)['image']
        
        return img.float(), l, p

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  LOAD SAMPLES FUNCTION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

def load_all_samples(dataset_name, config):
    """Load all samples for a dataset"""
    samples = []
    class_names = config['class_names']
    base_path = Path(config['path'])
    
    if not base_path.exists():
        print(f"⚠️ Path not found: {base_path}")
        return samples
    
    print(f"✅ Found path: {base_path}")
    
    # ==================== BRAIN TUMOR ====================
    if dataset_name == 'brain_tumor':
        for split in ['Training', 'Testing']:
            split_path = base_path / split
            if not split_path.exists():
                split_path = base_path / split.lower()
            if not split_path.exists():
                continue
            print(f"   Loading from: {split_path}")
            for class_dir in split_path.iterdir():
                if not class_dir.is_dir():
                    continue
                dir_name = class_dir.name.lower().replace('_', '')
                matched_class = None
                for cls in class_names:
                    if cls in dir_name or dir_name in cls:
                        matched_class = cls
                        break
                if matched_class is None:
                    continue
                count = 0
                for ext in ['.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG']:
                    for p in class_dir.rglob(f'*{ext}'):
                        samples.append((str(p), matched_class))
                        count += 1
                print(f"   {matched_class}: {count} images")
        return samples
    
    # ==================== FRACATLAS ====================
    elif dataset_name == 'fracatlas':
        print("   Scanning for FracAtlas images...")
        image_extensions = {'.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG', '.bmp', '.tiff'}
        
        class_folders = {}
        for item in base_path.iterdir():
            if item.is_dir():
                name_lower = item.name.lower()
                print(f"   Found folder: {item.name}")
                if 'fractured' in name_lower and 'not' not in name_lower and 'non' not in name_lower:
                    class_folders['Fractured'] = item
                elif 'not' in name_lower or 'non_fractured' in name_lower or 'normal' in name_lower or 'healthy' in name_lower:
                    class_folders['Not Fractured'] = item
        
        if class_folders:
            print(f"   Found class folders: {list(class_folders.keys())}")
            for class_name, folder_path in class_folders.items():
                count = 0
                for ext in image_extensions:
                    for img_path in folder_path.rglob(f'*{ext}'):
                        samples.append((str(img_path), class_name))
                        count += 1
                print(f"   {class_name}: {count} images")
        
        if not samples:
            print("   No class folders found, inferring from path...")
            for root, dirs, files in os.walk(base_path):
                current_dir = os.path.basename(root).lower()
                if 'fractured' in current_dir and 'not' not in current_dir and 'non' not in current_dir:
                    class_name = 'Fractured'
                elif 'not' in current_dir or 'non_fractured' in current_dir or 'normal' in current_dir or 'healthy' in current_dir:
                    class_name = 'Not Fractured'
                else:
                    class_name = None
                    path_parts = Path(root).parts
                    for part in reversed(path_parts):
                        part_lower = part.lower()
                        if 'fractured' in part_lower and 'not' not in part_lower and 'non' not in part_lower:
                            class_name = 'Fractured'
                            break
                        elif 'not' in part_lower or 'non_fractured' in part_lower or 'normal' in part_lower or 'healthy' in part_lower:
                            class_name = 'Not Fractured'
                            break
                
                if class_name is None:
                    continue
                
                for file in files:
                    if any(file.endswith(ext) for ext in image_extensions):
                        samples.append((os.path.join(root, file), class_name))
        
        if samples:
            print(f"   ✅ Loaded {len(samples)} FracAtlas images")
            class_counts = Counter([s[1] for s in samples])
            for cls, count in class_counts.items():
                print(f"   {cls}: {count}")
        return samples
    
    # ==================== BONE FRACTURE - COMPLETELY FIXED ====================
    elif dataset_name == 'bone_fracture':
        print("   Scanning for Bone Fracture images...")
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.JPG', '.JPEG', '.PNG'}
        
        # First, check if there are direct class folders
        class_folders = {}
        for item in base_path.iterdir():
            if item.is_dir():
                folder_name = item.name
                folder_lower = folder_name.lower()
                print(f"   Found folder: {folder_name}")
                
                # Check if this is a class folder
                if 'non_fractured' in folder_lower or 'nonfractured' in folder_lower:
                    class_folders['Not Fractured'] = item
                elif 'fractured' in folder_lower:
                    class_folders['Fractured'] = item
        
        # Load images from class folders
        if class_folders:
            print(f"\n   Found class folders: {list(class_folders.keys())}")
            for class_name, folder_path in class_folders.items():
                count = 0
                for ext in image_extensions:
                    for img_path in folder_path.rglob(f'*{ext}'):
                        samples.append((str(img_path), class_name))
                        count += 1
                print(f"   {class_name}: {count} images")
        else:
            print("   ⚠️ No direct class folders found!")
            print("   Checking for train/val/test splits...")
            
            # Fallback: Check train/val/test splits
            for split_name in ['train', 'val', 'test']:
                split_path = base_path / split_name
                if not split_path.exists():
                    continue
                
                print(f"   Found split: {split_name}")
                for class_dir in split_path.iterdir():
                    if not class_dir.is_dir():
                        continue
                    
                    folder_name = class_dir.name
                    folder_lower = folder_name.lower()
                    
                    if 'non_fractured' in folder_lower or 'nonfractured' in folder_lower:
                        cname = 'Not Fractured'
                    elif 'fractured' in folder_lower:
                        cname = 'Fractured'
                    else:
                        continue
                    
                    count = 0
                    for ext in image_extensions:
                        for img_path in class_dir.rglob(f'*{ext}'):
                            samples.append((str(img_path), cname))
                            count += 1
                    print(f"      {split_name}/{cname}: {count} images")
        
        # If still no samples, try to find any images and infer class from path
        if not samples:
            print("\n   Scanning all directories for images...")
            for ext in image_extensions:
                for img_path in base_path.rglob(f'*{ext}'):
                    # Try to determine class from path
                    path_str = str(img_path).lower()
                    if 'non_fractured' in path_str or 'nonfractured' in path_str:
                        samples.append((str(img_path), 'Not Fractured'))
                    elif 'fractured' in path_str:
                        samples.append((str(img_path), 'Fractured'))
                    else:
                        # Check parent folder
                        parent = img_path.parent.name.lower()
                        if 'non_fractured' in parent or 'nonfractured' in parent:
                            samples.append((str(img_path), 'Not Fractured'))
                        elif 'fractured' in parent:
                            samples.append((str(img_path), 'Fractured'))
                        else:
                            # Default based on containing folder
                            if 'fracture' in path_str:
                                samples.append((str(img_path), 'Fractured'))
                            else:
                                samples.append((str(img_path), 'Not Fractured'))
        
        # Summary
        if samples:
            print(f"\n   ✅ Loaded {len(samples)} Bone Fracture images")
            class_counts = Counter([s[1] for s in samples])
            for cls, count in class_counts.items():
                print(f"   {cls}: {count}")
            
            # Show sample paths for verification
            print("\n   📸 Sample paths:")
            for img_path, cls in samples[:5]:
                print(f"      {cls}: {Path(img_path).name}")
        else:
            print(f"\n   ❌ No images found!")
        
        return samples
    
    # ==================== TB CHEST ====================
    elif dataset_name == 'tb_chest':
        for folder in base_path.iterdir():
            if folder.is_dir():
                fn = folder.name.lower()
                if 'tb_normal' in fn:
                    cname = 'Normal'
                elif 'tb' in fn or 'tuberculosis' in fn:
                    cname = 'Tuberculosis'
                else:
                    continue
                count = 0
                for ext in ['.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG']:
                    for p in folder.rglob(f'*{ext}'):
                        samples.append((str(p), cname))
                        count += 1
                print(f"   {cname}: {count} images")
    
    # ==================== PNEUMONIA ====================
    elif dataset_name == 'pneumonia':
        for split in ['train', 'test', 'val']:
            split_path = base_path / split
            if not split_path.exists():
                continue
            print(f"   Loading from: {split_path}")
            for folder in split_path.iterdir():
                if folder.is_dir():
                    fn = folder.name.lower()
                    if 'normal_pneumonia' in fn:
                        cname = 'Normal'
                    elif 'pneumonia' in fn:
                        cname = 'Pneumonia'
                    else:
                        continue
                    count = 0
                    for ext in ['.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG']:
                        for p in folder.glob(f'*{ext}'):
                            samples.append((str(p), cname))
                            count += 1
                    print(f"   {cname}: {count} images")
    
    # ==================== DIABETIC RETINOPATHY ====================
    elif dataset_name == 'diabetic_retinopathy':
        folder_to_class = {
            '0': 'No_DR', '1': 'Mild', '2': 'Moderate', '3': 'Severe', '4': 'Proliferate_DR',
            'No_DR': 'No_DR', 'Mild': 'Mild', 'Moderate': 'Moderate',
            'Severe': 'Severe', 'Proliferate_DR': 'Proliferate_DR'
        }
        for folder in base_path.iterdir():
            if folder.is_dir():
                folder_name = folder.name.strip()
                cname = folder_to_class.get(folder_name, None)
                if cname is None:
                    for key in folder_to_class:
                        if key in folder_name:
                            cname = folder_to_class[key]
                            break
                if cname is None:
                    continue
                count = 0
                for ext in ['.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG']:
                    for p in folder.glob(f'*{ext}'):
                        samples.append((str(p), cname))
                        count += 1
                print(f"   {cname}: {count} images")
    
    # ==================== HAM10000 ====================
    elif dataset_name == 'ham10000':
        metadata_file = None
        for csv_file in base_path.glob('*.csv'):
            if 'metadata' in csv_file.name.lower() or 'HAM10000_metadata' in csv_file.name:
                metadata_file = csv_file
                break
        
        if metadata_file:
            df = pd.read_csv(metadata_file)
            img_paths = {}
            for ext in ['.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG']:
                for p in base_path.rglob(f'*{ext}'):
                    img_paths[p.stem] = str(p)
            for _, row in df.iterrows():
                img_id = row['image_id'] if 'image_id' in row else row.get('ImageID', '')
                label = row['dx'] if 'dx' in row else row.get('label', '')
                if img_id in img_paths and label in class_names:
                    samples.append((img_paths[img_id], label))
            print(f"   Loaded {len(samples)} images from metadata")
        else:
            print(f"⚠️ No metadata file found for HAM10000")
    
    if samples:
        print(f"✅ Total: {len(samples)} samples for {dataset_name}")
        class_counts = Counter([s[1] for s in samples])
        for cls, count in class_counts.items():
            print(f"   {cls}: {count}")
    else:
        print(f"⚠️ No samples loaded for {dataset_name}")
    
    return samples

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  GET TEST SPLIT - BRAIN TUMOR USES 70/30, OTHERS USE ORIGINAL                  ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

def get_test_split(dataset_name, samples, class_names):
    """Get test split - Brain Tumor uses 70/30, others use original 70/15/15"""
    
    if len(samples) == 0:
        return []
    
    # ✅ BRAIN TUMOR: 70% Train / 30% Test (NO VALIDATION)
    if dataset_name == 'brain_tumor':
        labels = [class_names.index(l) for _, l in samples]
        train_s, test_s = train_test_split(
            samples, 
            test_size=0.3, 
            stratify=labels, 
            random_state=SEED
        )
        print(f"   Train: {len(train_s)}, Test: {len(test_s)}")
        return test_s
    
    # ✅ OTHER DATASETS: Original 70/15/15 split
    else:
        labels = [class_names.index(l) for _, l in samples]
        train_s, temp_s = train_test_split(samples, test_size=0.3, stratify=labels, random_state=SEED)
        temp_labels = [class_names.index(l) for _, l in temp_s]
        val_s, test_s = train_test_split(temp_s, test_size=0.5, stratify=temp_labels, random_state=SEED)
        print(f"   Train: {len(train_s)}, Val: {len(val_s)}, Test: {len(test_s)}")
        return test_s

def load_dataset_with_exact_split(dataset_name, config):
    """Load test set with appropriate split"""
    print(f"\n📂 Loading {dataset_name}...")
    samples = load_all_samples(dataset_name, config)
    if not samples:
        print(f"⚠️ No samples found for {dataset_name}")
        return []
    
    class_names = config['class_names']
    test_s = get_test_split(dataset_name, samples, class_names)
    
    if not test_s:
        print(f"⚠️ No test samples for {dataset_name}")
        return []
    
    test_labels = [l for _, l in test_s]
    test_counts = Counter(test_labels)
    print(f"📊 Test set: {len(test_s)} samples")
    for cls in class_names:
        count = test_counts.get(cls, 0)
        print(f"   {cls}: {count} ({count/len(test_s)*100:.1f}%)")
    
    return test_s

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  LOAD MODELS                                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

def create_model_for_dataset(num_classes):
    """Create model with EXACT same architecture as training"""
    return EnhancedHViT(
        num_classes=num_classes,
        img_size=224,
        d_model=384,
        nhead=6,
        n_layers=4,
        dim_ffn=1536
    )

def load_models():
    """Load all 7 trained models"""
    models_dict = {}
    MODEL_BASE = Path('/kaggle/input/models/bhagawati4204/trained-models/pytorch/default/8')
    
    MODEL_FILE_MAP = {
        'brain_tumor': 'brain_tumor_best_model.pth',
        'fracatlas': 'fracatlas_best_model.pth',
        'tb_chest': 'tb_best_model.pth',
        'pneumonia': 'pneumonia_best_model.pth',
        'diabetic_retinopathy': 'DR_best_model.pth',
        'bone_fracture': 'bone_fracture_best_model.pth',
        'ham10000': 'skincancer_best_model.pth'
    }
    
    print("📂 Loading models...")
    
    for dataset_name, model_file in MODEL_FILE_MAP.items():
        model_path = MODEL_BASE / model_file
        if not model_path.exists():
            print(f"⚠️ Model not found: {model_path}")
            continue
        
        config = DATASET_CONFIGS[dataset_name]
        try:
            num_classes = len(config['class_names'])
            model = create_model_for_dataset(num_classes)
            
            checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)
            
            if isinstance(checkpoint, dict) and 'model' in checkpoint:
                state_dict = checkpoint['model']
            else:
                state_dict = checkpoint
            
            model.load_state_dict(state_dict, strict=False)
            model = model.to(DEVICE)
            model.eval()
            
            models_dict[dataset_name] = {'model': model, 'config': config}
            print(f"✅ Loaded {dataset_name} ({num_classes} classes)")
            
        except Exception as e:
            print(f"❌ Error loading {dataset_name}: {e}")
    
    return models_dict

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  EVALUATE MODEL                                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

def evaluate_model(model, dataset_name, config, samples, batch_size=32):
    """Evaluate model on test set"""
    class_names = config['class_names']
    is_grayscale = config['is_grayscale']
    num_classes = len(class_names)
    
    transform = A.Compose([
        A.Resize(height=224, width=224),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2()
    ])
    
    dataset = MedicalDataset(samples, class_names, transform, is_grayscale)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(loader, desc=f'Evaluating {dataset_name}', leave=False):
            images = images.to(DEVICE)
            with autocast():
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    if len(all_labels) == 0:
        return None
    
    results = {
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs,
        'class_names': class_names,
        'num_classes': num_classes,
        'accuracy': accuracy_score(all_labels, all_preds),
        'f1_weighted': f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        'f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(all_labels, all_preds),
        'mcc': matthews_corrcoef(all_labels, all_preds),
        'kappa': cohen_kappa_score(all_labels, all_preds),
        'test_samples': len(all_labels)
    }
    
    # Per-class metrics
    report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0)
    results['classification_report'] = report
    
    # Multi-class AUC
    if num_classes > 2 and len(np.unique(all_labels)) > 1:
        labels_bin = label_binarize(all_labels, classes=list(range(num_classes)))
        results['auc_roc'] = roc_auc_score(labels_bin, all_probs, multi_class='ovr', average='weighted')
    elif num_classes == 2 and len(np.unique(all_labels)) > 1:
        results['auc_roc'] = roc_auc_score(all_labels, all_probs[:, 1])
    else:
        results['auc_roc'] = 0.0
    
    confidence_scores = np.max(all_probs, axis=1)
    results['mean_confidence'] = np.mean(confidence_scores)
    results['num_errors'] = np.sum(all_preds != all_labels)
    
    return results

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  MAIN EVALUATION                                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

def main():
    print("="*80)
    print("🚀 EVALUATING ALL 7 DATASETS")
    print("="*80)
    print("\n📋 Split Configuration:")
    print("   - Brain Tumor: 70% Train / 30% Test (NO Validation)")
    print("   - Other Datasets: 70% Train / 15% Val / 15% Test")
    print(f"\n📂 Base path: {BASE_PATH}")
    
    models_dict = load_models()
    if not models_dict:
        print("❌ No models loaded!")
        return None
    
    all_results = {}
    for dataset_name, model_info in models_dict.items():
        print(f"\n{'='*60}")
        print(f"📊 Evaluating {dataset_name.upper()}")
        print(f"{'='*60}")
        
        config = model_info['config']
        model = model_info['model']
        samples = load_dataset_with_exact_split(dataset_name, config)
        
        if not samples:
            print(f"⚠️ No test samples for {dataset_name}")
            continue
        
        results = evaluate_model(model, dataset_name, config, samples)
        
        if results:
            all_results[dataset_name] = results
            print(f"\n✅ {config['name']}:")
            print(f"   Accuracy: {results['accuracy']:.4f}")
            print(f"   F1: {results['f1_weighted']:.4f}")
            print(f"   AUC: {results.get('auc_roc', 0):.4f}")
            print(f"   Test samples: {len(samples)}")
    
    return all_results

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  RUN EVALUATION                                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

all_results = main()

if all_results:
    print("\n" + "="*80)
    print("📊 FINAL SUMMARY")
    print("="*80)
    
    summary_data = []
    for name, results in all_results.items():
        config = DATASET_CONFIGS[name]
        summary_data.append({
            'Dataset': config['name'],
            'Accuracy': f"{results['accuracy']:.4f}",
            'F1': f"{results['f1_weighted']:.4f}",
            'AUC': f"{results.get('auc_roc', 0):.4f}",
            'Samples': results['test_samples']
        })
    
    df_summary = pd.DataFrame(summary_data)
    print(df_summary.to_string(index=False))
    
    avg_acc = np.mean([r['accuracy'] for r in all_results.values()])
    avg_f1 = np.mean([r['f1_weighted'] for r in all_results.values()])
    print(f"\n📊 Average Performance: Accuracy={avg_acc:.4f}, F1={avg_f1:.4f}")
    
    print("\n" + "="*80)
    print("✅ EVALUATION COMPLETE!")
    print("="*80)
else:
    print("❌ No results to display")

🚀 EVALUATING ALL 7 DATASETS

📋 Split Configuration:
   - Brain Tumor: 70% Train / 30% Test (NO Validation)
   - Other Datasets: 70% Train / 15% Val / 15% Test

📂 Base path: /kaggle/input/datasets/bhagawati4204/daataasets
📂 Loading models...
✅ Loaded brain_tumor (4 classes)
✅ Loaded fracatlas (2 classes)
✅ Loaded tb_chest (2 classes)
✅ Loaded pneumonia (2 classes)
✅ Loaded diabetic_retinopathy (5 classes)
✅ Loaded bone_fracture (2 classes)
✅ Loaded ham10000 (7 classes)

📊 Evaluating BRAIN_TUMOR

📂 Loading brain_tumor...
✅ Found path: /kaggle/input/datasets/bhagawati4204/daataasets/Brain Tumor MRI Dataset/Brain Tumor MRI Dataset
   Loading from: /kaggle/input/datasets/bhagawati4204/daataasets/Brain Tumor MRI Dataset/Brain Tumor MRI Dataset/Training
   pituitary: 1400 images
   notumor: 1400 images
   meningioma: 1400 images
   glioma: 1400 images
   Loading from: /kaggle/input/datasets/bhagawati4204/daataasets/Brain Tumor MRI Dataset/Brain Tumor MRI Dataset/Testing
   pituitary: 400 imag

Evaluating brain_tumor:   0%|          | 0/68 [00:00<?, ?it/s]


✅ Brain Tumor MRI:
   Accuracy: 0.9866
   F1: 0.9866
   AUC: 0.9989
   Test samples: 2160

📊 Evaluating FRACATLAS

📂 Loading fracatlas...
✅ Found path: /kaggle/input/datasets/bhagawati4204/daataasets/Fracture Dataset/Fracture Dataset/FracAtlas/images
   Scanning for FracAtlas images...
   Found folder: Non_fractured
   Found folder: Fractured
   Found class folders: ['Not Fractured', 'Fractured']
   Not Fractured: 3366 images
   Fractured: 717 images
   ✅ Loaded 4083 FracAtlas images
   Not Fractured: 3366
   Fractured: 717
   Train: 2858, Val: 612, Test: 613
📊 Test set: 613 samples
   Not Fractured: 505 (82.4%)
   Fractured: 108 (17.6%)


Evaluating fracatlas:   0%|          | 0/20 [00:00<?, ?it/s]

Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file


...

Evaluating bone_fracture:   0%|          | 0/50 [00:00<?, ?it/s]

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file



✅ Bone Fracture Binary:
   Accuracy: 0.9987
   F1: 0.9987
   AUC: 0.9997
   Test samples: 1587

📊 Evaluating HAM10000

📂 Loading ham10000...
✅ Found path: /kaggle/input/datasets/bhagawati4204/daataasets/skin cancer/skin cancer
   Loaded 10015 images from metadata
✅ Total: 10015 samples for ham10000
   bkl: 1099
   nv: 6705
   df: 115
   mel: 1113
   vasc: 142
   bcc: 514
   akiec: 327
   Train: 7010, Val: 1502, Test: 1503
📊 Test set: 1503 samples
   akiec: 49 (3.3%)
   bcc: 77 (5.1%)
   bkl: 165 (11.0%)
   df: 17 (1.1%)
   mel: 167 (11.1%)
   nv: 1006 (66.9%)
   vasc: 22 (1.5%)


Evaluating ham10000:   0%|          | 0/47 [00:00<?, ?it/s]


✅ HAM10000 Skin Cancer:
   Accuracy: 0.8436
   F1: 0.8499
   AUC: 0.9668
   Test samples: 1503

📊 FINAL SUMMARY
              Dataset Accuracy     F1    AUC  Samples
      Brain Tumor MRI   0.9866 0.9866 0.9989     2160
            FracAtlas   0.8662 0.8601 0.8582      613
       TB Chest X-Ray   0.9698 0.9701 0.9940      630
Pneumonia Chest X-Ray   0.9454 0.9457 0.9858      879
 Diabetic Retinopathy   0.8127 0.8133 0.9642      550
 Bone Fracture Binary   0.9987 0.9987 0.9997     1587
 HAM10000 Skin Cancer   0.8436 0.8499 0.9668     1503

📊 Average Performance: Accuracy=0.9176, F1=0.9178

✅ EVALUATION COMPLETE!
